In [5]:
from pathlib import Path
import pandas as pd
import sys
!{sys.executable} -m pip install plotly streamlit
import plotly.express as px
import streamlit as st

FILE = "Water Quality.csv"  # Set to full path if needed
st.set_page_config(page_title="AquaInsight Water Quality Explorer", layout="wide")
st.title("AquaInsight: Water Quality Explorer")
st.caption("Explore recorded measurements over time and across monitoring locations.")

@st.cache_data
def load_data():
    df = pd.read_csv(FILE, low_memory=False)
    req = {"ActivityStartDate", "CharacteristicName", "ResultValue"}
    if req-set(df.columns): raise ValueError(f"Missing columns: {req-set(df.columns)}")
    df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce")
    df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
    df = df.dropna(subset=["ActivityStartDate", "CharacteristicName", "ResultValue"])
    for c in ["MonitoringLocationID", "MonitoringLocationName"]:
        if c not in df: df[c] = "Unknown"
        df[c] = df[c].fillna("Unknown").astype(str)
    return df

if not Path(FILE).exists():
    st.error(f"Cannot find {FILE}. Edit FILE near the top to the full CSV path."); st.stop()
try: df = load_data()
except Exception as e: st.error(str(e)); st.stop()

with st.sidebar:
    st.header("Filters")
    params = sorted(df.CharacteristicName.astype(str).unique())
    selected_params = st.multiselect("Parameter", params, default=params[:min(3,len(params))])
    locs = sorted(df.MonitoringLocationID.unique())
    selected_locs = st.multiselect("Monitoring location ID", locs, default=locs[:min(10,len(locs))])
    start, end = df.ActivityStartDate.min().date(), df.ActivityStartDate.max().date()
    dates = st.date_input("Date range", value=(start,end), min_value=start, max_value=end)
    agg = st.selectbox("Aggregation", ["Monthly median","Monthly mean","Daily median"])

if not selected_params or not selected_locs or not isinstance(dates,(tuple,list)) or len(dates)!=2:
    st.info("Select at least one parameter, one location, and a start/end date."); st.stop()
f = df[df.CharacteristicName.astype(str).isin(selected_params) & df.MonitoringLocationID.isin(selected_locs)]
f = f[(f.ActivityStartDate.dt.date >= dates[0]) & (f.ActivityStartDate.dt.date <= dates[1])]
if f.empty: st.warning("No records match the filters."); st.stop()
a,b,c = st.columns(3)
a.metric("Measurements", f"{len(f):,}"); b.metric("Parameters", f.CharacteristicName.nunique()); c.metric("Locations", f.MonitoringLocationID.nunique())
daily = agg == "Daily median"
f["Time"] = f.ActivityStartDate.dt.floor("D") if daily else f.ActivityStartDate.dt.to_period("M").dt.to_timestamp()
stat = "mean" if agg == "Monthly mean" else "median"
series = f.groupby(["Time","CharacteristicName","MonitoringLocationID"],as_index=False).ResultValue.agg(stat)
st.subheader("Measurements over time")
fig = px.line(series,x="Time",y="ResultValue",color="CharacteristicName",line_group="MonitoringLocationID",
              hover_data=["MonitoringLocationID"],markers=True,title=agg)
fig.update_layout(yaxis_title="ResultValue (original units)")
st.plotly_chart(fig,use_container_width=True)
left,right = st.columns(2)
with left:
    st.subheader("Parameter distributions")
    st.plotly_chart(px.box(f,x="CharacteristicName",y="ResultValue",points="outliers"),use_container_width=True)
with right:
    st.subheader("Measurement counts by location")
    counts = f.groupby("MonitoringLocationID",as_index=False).size()
    st.plotly_chart(px.bar(counts,x="MonitoringLocationID",y="size"),use_container_width=True)
st.subheader("Filtered records")
st.dataframe(f.sort_values("ActivityStartDate",ascending=False),use_container_width=True,height=350)
st.download_button("Download filtered CSV",f.to_csv(index=False).encode("utf-8"),"aquainsight_filtered_data.csv","text/csv")



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2026-10-19 18:54:47.151 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 18:54:47.153 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 18:54:47.156 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 18:54:47.157 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 18:54:47.160 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 18:54:47.161 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 18:54:47.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 18:54:47.164 No runtime found, using MemoryCacheStorageManager
2026-10-19 18:54:47.433 Thread 'MainThread':

False